# 23｜強化學習_策略價值與深度RL

以兩臂 bandit 示範探索、回饋與增量式價值估計。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])


In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.gcf().savefig(folder / f"{name}.svg", bbox_inches="tight")
    plt.show()

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

rng = np.random.default_rng(SEED)


## 課堂小實驗

In [ ]:
true_means = np.array([0.2, 0.8])
Q = np.zeros(2); counts = np.zeros(2, dtype=int); rewards = []
epsilon = 0.12
for step in range(300):
    action = rng.integers(2) if rng.random() < epsilon else int(np.argmax(Q))
    reward = rng.normal(true_means[action], 0.25)
    counts[action] += 1
    Q[action] += (reward - Q[action]) / counts[action]
    rewards.append(reward)
pd.DataFrame({'arm': [0,1], 'pulls': counts, 'estimated_value': Q, 'true_mean': true_means}).round(3)

## 執行結果圖

In [ ]:
rewards = np.array(rewards)
moving = np.convolve(rewards, np.ones(25)/25, mode='valid')
plt.plot(np.arange(24, len(rewards)), moving)
plt.axhline(true_means.max(), color='tab:red', ls='--', label='best arm mean')
plt.xlabel('interaction'); plt.ylabel('25-step mean reward'); plt.title('Exploration gradually identifies the better action')
plt.legend()
savefig('23_rl_demo')
report('reinforcement_learning', {'counts': counts.tolist(), 'Q': Q.tolist(), 'mean_reward': rewards.mean()})

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。